# 🎬 Mazinger Studio

**Dub any video into another language with AI** — powered by [Mazinger](https://pypi.org/project/mazinger/).

Paste a YouTube URL (or upload a file), pick a language and voice, then hit **Start Dubbing**.

### How to use

1. **Run Step 1** below to install dependencies (~2 min)
2. **Run Step 2** to download & prepare models (~5-10 min first time)
3. **Run Step 3** to launch the app — a public link will appear
4. **Open the link** and start dubbing!

### What you need

| Requirement | Details |
|---|---|
| **GPU runtime** *(recommended)* | For voice synthesis — *Runtime → Change runtime type → T4 GPU* |
| **OpenAI API key** *(optional)* | Only needed if you choose OpenAI as LLM provider. By default we use **Ollama** (local, free). |


In [1]:
#@title 📦 Step 1: Install Dependencies { display-mode: "form" }

#@markdown **Choose your TTS (voice synthesis) engine:**
tts_engine = "Qwen (recommended)" #@param ["Qwen (recommended)", "Chatterbox"]

import subprocess, sys, shutil, time

# ── Ensure pip is available ──────────────────────────────────────
try:
    subprocess.check_call([sys.executable, "-m", "pip", "--version"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
except subprocess.CalledProcessError:
    print("📥 Bootstrapping pip...")
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"],
                          stdout=subprocess.DEVNULL)

# ── Python packages ──────────────────────────────────────────────
if "Chatterbox" in tts_engine:
    extras = "tts-chatterbox,transcribe-faster"
    print("Installing Mazinger with Chatterbox TTS + Faster Whisper...")
else:
    extras = "tts,transcribe-faster"
    print("Installing Mazinger with Qwen TTS + Faster Whisper...")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                        f"mazinger[{extras}]", "gradio>=4.0"])

# ── Ollama (local LLM server) ───────────────────────────────────
if not shutil.which("ollama"):
    print("\n📥 Installing Ollama (local LLM server)...")
    subprocess.check_call(
        ["bash", "-c", "curl -fsSL https://ollama.com/install.sh | sh"],
        stdout=subprocess.DEVNULL,
    )
    print("✅ Ollama installed")
else:
    print("\n✅ Ollama already installed")

# Start Ollama server in background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
print("✅ Ollama server started (model will be pulled on first run)")

# ── GPU check ────────────────────────────────────────────────────
try:
    import torch
    if torch.cuda.is_available():
        print(f"\n✅ GPU detected: {torch.cuda.get_device_name(0)}")
    else:
        print("\n⚠️  No GPU detected — voice synthesis will be slow.")
        print("   Tip: Runtime → Change runtime type → T4 GPU")
except ImportError:
    pass

print("\n✅ Ready! Run the next cell to download & prepare models.")

Installing Mazinger with Qwen TTS + Faster Whisper...



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip



✅ Ollama already installed
✅ Ollama server started (model will be pulled on first run)

✅ GPU detected: NVIDIA GeForce RTX 4060 Ti

✅ Ready! Run the next cell to download & prepare models.


In [2]:
#@title 📥 Step 2: Download & Prepare Models { display-mode: "form" }

#@markdown Downloads all required models to disk so they're ready when needed.
#@markdown This avoids long waits during your first dubbing run.
#@markdown
#@markdown **Expected download sizes:**
#@markdown - Ollama LLM model: ~1.5 GB
#@markdown - Faster-Whisper (large-v3): ~3 GB
#@markdown - Qwen TTS / Chatterbox: ~3-4 GB

import subprocess, sys, os, time

print("📥 Downloading models (this may take 5-10 min on first run)...\n")

# ── 1. Ollama LLM model ─────────────────────────────────────────
print("1️⃣  Pulling Ollama LLM model (qwen3.5:2b-q8_0)...")
try:
    result = subprocess.run(
        ["ollama", "pull", "qwen3.5:2b-q8_0"],
        timeout=600,
    )
    if result.returncode == 0:
        print("   ✅ Ollama model ready\n")
    else:
        print("   ⚠️  Ollama pull had issues (model will be pulled on first run)\n")
except Exception as e:
    print(f"   ⚠️  Ollama pull failed: {e}\n")

# ── 2. Faster-Whisper model ─────────────────────────────────────
print("2️⃣  Downloading Faster-Whisper model (large-v3)...")
try:
    from huggingface_hub import snapshot_download
    snapshot_download("Systran/faster-whisper-large-v3")
    print("   ✅ Faster-Whisper model ready\n")
except Exception as e:
    print(f"   ⚠️  Faster-Whisper download failed: {e}\n")

# ── 3. TTS model ────────────────────────────────────────────────
# `tts_engine` was set by Step 1
if "Chatterbox" in tts_engine:
    print("3️⃣  Downloading Chatterbox TTS model...")
    try:
        from huggingface_hub import snapshot_download
        snapshot_download("ResembleAI/chatterbox")
        print("   ✅ Chatterbox model ready\n")
    except Exception as e:
        print(f"   ⚠️  Chatterbox download failed: {e}\n")
else:
    print("3️⃣  Downloading Qwen TTS model...")
    try:
        from huggingface_hub import snapshot_download
        snapshot_download("Qwen/Qwen3-TTS-12Hz-1.7B-Base")
        print("   ✅ Qwen TTS model ready\n")
    except Exception as e:
        print(f"   ⚠️  Qwen TTS download failed: {e}\n")

print("✅ All models downloaded and cached! Run the next cell to launch the app.")

📥 Downloading models (this may take 5-10 min on first run)...

1️⃣  Pulling Ollama LLM model (qwen3.5:2b-q8_0)...


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling b709d81508a0: 100% ▕██████████████████▏ 2.7 GB                         
pulling 9be69ef46306: 100% ▕██████████████████▏  11 KB                         
pulling 9371364b27a5: 100% ▕██████████████████▏   65 B                         
pulling ee043a99abe5: 100% ▕██████████████████▏  473 B                         
verifying sha256 digest 
writing manifest 
success 
/workspace/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


   ✅ Ollama model ready

2️⃣  Downloading Faster-Whisper model (large-v3)...


Fetching 7 files: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 45100.04it/s]


   ✅ Faster-Whisper model ready

3️⃣  Downloading Qwen TTS model...


Fetching 13 files: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 13/13 [00:00<00:00, 60116.82it/s]

   ✅ Qwen TTS model ready

✅ All models downloaded and cached! Run the next cell to launch the app.


In [ ]:
#@title 🚀 Step 3: Launch Mazinger Studio { display-mode: "form" }

import subprocess, os

# ── Fetch studio scripts from GitHub ─────────────────────────────
_BASE = "https://raw.githubusercontent.com/bakrianoo/mazinger/refs/heads/master/docs/notebooks/studio"
_SCRIPTS = ["constants.py", "theme.py", "helpers.py", "pipeline.py", "app.py"]

os.makedirs("studio", exist_ok=True)
for _script in _SCRIPTS:
    _dest = os.path.join("studio", _script)
    if not os.path.exists(_dest):
        subprocess.check_call(["wget", "-q", f"{_BASE}/{_script}", "-O", _dest])
        print(f"  ✅ Downloaded {_script}")
    else:
        print(f"  ✔ {_script} (cached)")

print("\n🚀 Launching Mazinger Studio...\n")

# ── Run the app ──────────────────────────────────────────────────
subprocess.check_call(["python", "studio/app.py"])

  ✔ constants.py (cached)
  ✔ theme.py (cached)
  ✔ helpers.py (cached)
  ✔ pipeline.py (cached)
  ✔ app.py (cached)

🚀 Launching Mazinger Studio...



/workspace/mazinger/docs/notebooks/studio/app.py:14: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, title="Mazinger Studio", css=CSS) as app:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://bfdafaa0e75d8977e0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR: [youtube] k38EASGY3F0: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies
